# PG-Stack · Step 3 — Flat Stacking Baselines + Evaluation Toolkit

**Goal:** build the strongest *non-graph* stacking systems. PG-Stack (Step 4) must beat these.

| Model | Meta-learner input | Role |
|---|---|---|
| `stack_lr` | 3 base predictions (tab_lesion, emb_lr, cnn) | classic stacking |
| `stack_xgb` | same | classic stacking, non-linear |
| `stack_xgb_pctx` | + **hand-crafted patient context** of each prediction (rank / z-score / gap to patient max) | strongest non-graph patient-aware stack — **main competitor of PG-Stack** |
| `stack_xgb_all` | + tab_pnorm prediction | practical upper reference |

**Evaluation:** pAUC80 (official), AUROC, sens@spec95, spec@sens95, **top-5/10 recall per patient**, fold mean ± SD, **patient-level bootstrap 95% CI**, **paired bootstrap test** vs baselines.

**Runs in the same session** — reads `/kaggle/working/oof_all.csv` from Step 2, writes back to `/kaggle/working`. CPU is enough (~10–20 min, mostly bootstrap).

In [ ]:
import os, sys, time, warnings
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
SEED = 42
N_BOOT = 300            # patient-level bootstrap resamples (raise to 1000 for the final paper)
assert os.path.exists(f"{WORK}/oof_all.csv"), "oof_all.csv missing - run Step 2 first in this session"

## 1. Shared evaluation toolkit → `/kaggle/working/pgstack_eval.py` (reused by Steps 4–6)

In [ ]:
%%writefile /kaggle/working/pgstack_eval.py
# pgstack_eval.py — shared evaluation toolkit for all PG-Stack steps
import numpy as np, pandas as pd
from sklearn.metrics import roc_curve, auc, roc_auc_score, brier_score_loss


def pauc_80(y_true, y_pred, min_tpr=0.80):
    """Official ISIC 2024 metric: partial AUC above min_tpr. Range [0, 0.2]."""
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = -1.0 * np.asarray(y_pred)
    max_fpr = abs(1 - min_tpr)
    fpr, tpr, _ = roc_curve(v_gt, v_pred)
    stop = np.searchsorted(fpr, max_fpr, "right")
    tpr = np.append(tpr[:stop], np.interp(max_fpr, [fpr[stop - 1], fpr[stop]], [tpr[stop - 1], tpr[stop]]))
    fpr = np.append(fpr[:stop], max_fpr)
    return auc(fpr, tpr)


def sens_at_spec(y, p, spec=0.95):
    y, p = np.asarray(y), np.asarray(p)
    neg = np.sort(p[y == 0]); thr = neg[min(int(spec * len(neg)), len(neg) - 1)]
    return float((p[y == 1] > thr).mean())


def spec_at_sens(y, p, sens=0.95):
    y, p = np.asarray(y), np.asarray(p)
    pos = np.sort(p[y == 1]); thr = pos[int((1 - sens) * len(pos))]
    return float((p[y == 0] < thr).mean())


def topk_recall(y, p, patient, k=10):
    """Share of all cancers that fall inside each patient's top-k ranked lesions (triage view)."""
    d = pd.DataFrame(dict(y=np.asarray(y), p=np.asarray(p), pid=np.asarray(patient)))
    d["r"] = d.groupby("pid")["p"].rank(ascending=False, method="first")
    return float(d.loc[d.y == 1, "r"].le(k).mean())


def ece(y, p, n_bins=10):
    """Expected calibration error with equal-frequency bins (only meaningful for probabilities)."""
    y, p = np.asarray(y), np.clip(np.asarray(p), 0, 1)
    order = np.argsort(p); bins = np.array_split(order, n_bins)
    return float(sum(len(b) / len(p) * abs(y[b].mean() - p[b].mean()) for b in bins))


def full_report(name, y, p, patient, is_prob=False):
    r = dict(model=name, pauc80=pauc_80(y, p), auroc=roc_auc_score(y, p),
             sens_spec95=sens_at_spec(y, p, 0.95), spec_sens95=spec_at_sens(y, p, 0.95),
             top5_recall=topk_recall(y, p, patient, 5), top10_recall=topk_recall(y, p, patient, 10))
    if is_prob:
        r["ece"] = ece(y, p); r["brier"] = brier_score_loss(y, np.clip(p, 0, 1))
    return r


def fold_stats(y, p, folds, metric=pauc_80):
    v = [metric(y[folds == f], p[folds == f]) for f in np.unique(folds)]
    return float(np.mean(v)), float(np.std(v))


class PatientBootstrap:
    """Resample whole patients (not lesions). Same resamples are reused for every model -> paired tests."""
    def __init__(self, patient, n_boot=300, seed=0):
        pid = pd.Series(np.asarray(patient))
        g = pid.groupby(pid).indices
        self.groups = [np.asarray(v, dtype=np.int32) for v in g.values()]
        rng = np.random.default_rng(seed)
        # store only the patient picks (small); row indices are rebuilt on the fly (saves RAM)
        self.picks = rng.integers(0, len(self.groups), (n_boot, len(self.groups)))

    def _idx(self, pick):
        return np.concatenate([self.groups[i] for i in pick])

    @property
    def samples(self):
        for pick in self.picks:
            yield self._idx(pick)

    def scores(self, y, p, metric=pauc_80):
        y, p = np.asarray(y), np.asarray(p)
        out = []
        for idx in self.samples:
            if y[idx].sum() == 0: continue
            out.append(metric(y[idx], p[idx]))
        return np.array(out)

    def ci(self, y, p, metric=pauc_80):
        s = self.scores(y, p, metric)
        return float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

    def paired_test(self, y, p_new, p_base, metric=pauc_80):
        """Returns (mean difference, 95% CI of difference, one-sided p-value that new <= base)."""
        y = np.asarray(y); a, b = np.asarray(p_new), np.asarray(p_base); d = []
        for idx in self.samples:
            if y[idx].sum() == 0: continue
            d.append(metric(y[idx], a[idx]) - metric(y[idx], b[idx]))
        d = np.array(d)
        return float(d.mean()), (float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))), float((d <= 0).mean())


In [ ]:
sys.path.insert(0, WORK)
import importlib, pgstack_eval; importlib.reload(pgstack_eval)
from pgstack_eval import *

## 2. Load base-learner OOF predictions

In [ ]:
oof = pd.read_csv(f"{WORK}/oof_all.csv")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = len(np.unique(folds))
BASE = ["oof_tab_lesion", "oof_emb_lr", "oof_cnn"]
assert oof[BASE].notna().all().all(), "some base predictions are missing (did all CNN folds finish?)"
print(oof.shape, "| positives:", y.sum(), "| patients:", oof.patient_id.nunique())

## 3. Meta-features
- **logits** of each base prediction (puts LR/CNN/GBDT outputs on one scale)
- **hand-crafted patient context** of each prediction: percentile rank within patient, z-score within patient, gap to the patient's maximum, number of lesions → the *manual* ugly-duckling signal at the meta level

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

M = pd.DataFrame(index=oof.index)
for c in BASE + ["oof_tab_pnorm"]:
    M[f"{c}_logit"] = logit(oof[c].values)

g = oof.groupby("patient_id")
for c in BASE:
    M[f"{c}_prank"] = g[c].rank(pct=True)
    M[f"{c}_pz"]    = (oof[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6)
    M[f"{c}_pgap"]  = M[f"{c}_logit"] - logit(g[c].transform("max").values)
M["n_lesions"] = g["isic_id"].transform("size")

F_BASE = [f"{c}_logit" for c in BASE]
F_PCTX = F_BASE + [f"{c}_{s}" for c in BASE for s in ["prank", "pz", "pgap"]] + ["n_lesions"]
F_ALL  = F_PCTX + ["oof_tab_pnorm_logit"]
print(len(F_BASE), len(F_PCTX), len(F_ALL))

## 4. Meta-learners, trained with the **same patient folds**
For fold *f*: train the meta-learner on the OOF predictions of the other folds, predict fold *f* (standard OOF stacking).

In [ ]:
XGB_PARAMS = dict(n_estimators=400, learning_rate=0.03, max_depth=3, min_child_weight=5,
                  subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, scale_pos_weight=20, tree_method="hist",
                  eval_metric="auc", random_state=SEED, n_jobs=-1)

def run_meta(feats, kind):
    pred = np.zeros(len(oof))
    for f in range(N_FOLDS):
        tr, va = folds != f, folds == f
        X_tr, X_va = M.loc[tr, feats].values, M.loc[va, feats].values
        if kind == "lr":
            sc = StandardScaler().fit(X_tr)
            m = LogisticRegression(C=1.0, max_iter=2000).fit(sc.transform(X_tr), y[tr])
            pred[va] = m.predict_proba(sc.transform(X_va))[:, 1]
        else:
            m = xgb.XGBClassifier(**XGB_PARAMS).fit(X_tr, y[tr])
            pred[va] = m.predict_proba(X_va)[:, 1]
    return pred

t0 = time.time()
oof["stack_lr"]       = run_meta(F_BASE, "lr")
oof["stack_xgb"]      = run_meta(F_BASE, "xgb")
oof["stack_xgb_pctx"] = run_meta(F_PCTX, "xgb")
oof["stack_xgb_all"]  = run_meta(F_ALL,  "xgb")
print(f"meta-learners done in {time.time()-t0:.0f}s")

## 5. Results table (point estimates + fold mean ± SD)

In [ ]:
MODELS = {"tab_lesion": "oof_tab_lesion", "tab_pnorm": "oof_tab_pnorm", "emb_lr": "oof_emb_lr",
          "cnn": "oof_cnn", "avg_tab_img": "avg_tab_img", "stack_lr": "stack_lr",
          "stack_xgb": "stack_xgb", "stack_xgb_pctx": "stack_xgb_pctx", "stack_xgb_all": "stack_xgb_all"}
PROB_MODELS = {"stack_lr", "stack_xgb", "stack_xgb_pctx", "stack_xgb_all"}   # trained on full data = real probabilities

rows = []
for name, col in MODELS.items():
    r = full_report(name, y, oof[col].values, pid, is_prob=(name in PROB_MODELS))
    mu, sd = fold_stats(y, oof[col].values, folds)
    r["pauc80_fold"] = f"{mu:.4f} ± {sd:.4f}"
    rows.append(r)
res = pd.DataFrame(rows).set_index("model")
pd.set_option("display.width", 200)
res.round(4)

## 6. Patient-level bootstrap: 95% CI + paired significance tests

In [ ]:
t0 = time.time()
boot = PatientBootstrap(pid, n_boot=N_BOOT, seed=SEED)

ci = {}
for name in ["tab_lesion", "tab_pnorm", "stack_lr", "stack_xgb", "stack_xgb_pctx", "stack_xgb_all"]:
    lo, hi = boot.ci(y, oof[MODELS[name]].values)
    ci[name] = f"[{lo:.4f}, {hi:.4f}]"
res["pauc80_95CI"] = pd.Series(ci)
print(f"CIs done in {time.time()-t0:.0f}s")

tests = []
for new, base in [("stack_xgb", "tab_lesion"), ("stack_xgb_pctx", "stack_xgb"),
                  ("stack_xgb_pctx", "tab_pnorm"), ("stack_xgb_all", "tab_pnorm")]:
    d, (lo, hi), p = boot.paired_test(y, oof[MODELS[new]].values, oof[MODELS[base]].values)
    tests.append(dict(comparison=f"{new} vs {base}", delta_pauc=round(d, 4),
                      ci95=f"[{lo:+.4f}, {hi:+.4f}]", p_value=round(p, 4)))
tests = pd.DataFrame(tests)
print(f"bootstrap total {time.time()-t0:.0f}s")
tests

In [ ]:
show = ["pauc80", "pauc80_95CI", "pauc80_fold", "auroc", "sens_spec95", "spec_sens95",
        "top5_recall", "top10_recall", "ece", "brier"]
final = res[show]
print(final.round(4).to_string())

## 7. Save

In [ ]:
keep = ["isic_id", "patient_id", "fold", "target"] + BASE + ["oof_tab_pnorm", "avg_tab_img",
        "stack_lr", "stack_xgb", "stack_xgb_pctx", "stack_xgb_all"]
oof[keep].to_csv(f"{WORK}/oof_step3.csv", index=False)
final.to_csv(f"{WORK}/results_step3.csv")
tests.to_csv(f"{WORK}/tests_step3.csv", index=False)
print(sorted(f for f in os.listdir(WORK) if not f.startswith(".")))